# Semantic Model Maintenance

Manages semantic model settings that can't be changed in the Fabric UI when
the model is owned by a Service Principal.

**Capabilities:**
1. **View current refresh schedules** for all models
2. **Set / update refresh schedules** (daily/weekly, times, timezone)
3. **View ownership** and optionally **transfer ownership** (takeover)
4. **View refresh history** for troubleshooting

**Why this notebook?** When a semantic model is owned by a Service Principal,
the Fabric UI doesn't show the refresh schedule settings. This notebook
uses the Power BI REST API directly to configure schedules.

> **API used:** `PATCH /v1.0/myorg/groups/{groupId}/datasets/{datasetId}/refreshSchedule`

## 1. Configuration

In [ ]:
# === CONFIGURATION ===
WORKSPACE_ID = "68c48a5c-8b88-4559-962d-5dc7f29ab96a"  # Target workspace

# Exclude models whose names start with any of these prefixes (case-sensitive)
EXCLUDE_PREFIXES = ["BACKUP_"]  # e.g., ["BACKUP_", "OLD_", "ARCHIVE_"]

PBI_API = "https://api.powerbi.com/v1.0/myorg"

print(f"Target workspace: {WORKSPACE_ID}")
print(f"Excluded prefixes: {EXCLUDE_PREFIXES}")

## 2. Authentication

In [ ]:
import requests
import json
import time
from datetime import datetime

try:
    from notebookutils import mssparkutils
    access_token = mssparkutils.credentials.getToken("https://analysis.windows.net/powerbi/api")
    print("\u2713 Authenticated via Fabric notebook token")
except ImportError:
    from azure.identity import DefaultAzureCredential
    credential = DefaultAzureCredential()
    token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
    access_token = token.token
    print("\u2713 Authenticated via DefaultAzureCredential")

headers = {
    "Authorization": f"Bearer {access_token}",
    "Content-Type": "application/json"
}

def api_get(url):
    resp = requests.get(url, headers=headers, timeout=60)
    if resp.ok:
        return resp.json()
    print(f"  \u2717 GET {resp.status_code}: {resp.text[:300]}")
    return None

def api_post(url, body=None):
    return requests.post(url, headers=headers, json=body, timeout=60)

def api_patch(url, body=None):
    return requests.patch(url, headers=headers, json=body, timeout=60)

ws_info = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}")
if ws_info:
    print(f"\u2713 Connected to workspace: {ws_info.get('name', 'unknown')}")

me = api_get(f"{PBI_API}/profile")
if me:
    print(f"\u2713 Running as: {me.get('displayName', me.get('emailAddress', 'unknown'))}")
else:
    print("\u2139\ufe0f Running as service principal (no user profile)")

## 3. List semantic models and their current refresh schedules

In [ ]:
# Get all datasets in the workspace
datasets_data = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets")
all_models = datasets_data.get("value", []) if datasets_data else []

# Exclude models matching EXCLUDE_PREFIXES
excluded = [m for m in all_models if any(m["name"].startswith(p) for p in EXCLUDE_PREFIXES)]
all_models = [m for m in all_models if not any(m["name"].startswith(p) for p in EXCLUDE_PREFIXES)]

if excluded:
    print(f"Excluded by prefix ({len(excluded)}):")
    for m in excluded:
        print(f"  [EXCLUDED] {m['name']}")
    print()

print(f"Found {len(all_models)} semantic model(s):\n")
print(f"{'Name':<40} {'Owner':<30} {'Refreshable':<12} {'ID'}")
print("-" * 110)

refreshable_models = []
for m in sorted(all_models, key=lambda x: x["name"]):
    name = m["name"]
    owner = m.get("configuredBy", "unknown")
    refreshable = m.get("isRefreshable", False)
    mid = m["id"]
    flag = "\u2713" if refreshable else "-"
    print(f"  {name:<38} {owner:<30} {flag:<12} {mid}")
    if refreshable:
        refreshable_models.append(m)

print(f"\n{len(refreshable_models)} refreshable model(s)")

In [ ]:
# Show current refresh schedule for each refreshable model
print("CURRENT REFRESH SCHEDULES")
print("=" * 70)

model_schedules = {}  # name -> schedule dict

for m in refreshable_models:
    name = m["name"]
    mid = m["id"]

    schedule = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/refreshSchedule")
    model_schedules[name] = {"id": mid, "schedule": schedule}

    print(f"\n{name}:")
    if schedule:
        enabled = schedule.get("enabled", False)
        days = schedule.get("days", [])
        times = schedule.get("times", [])
        tz = schedule.get("localTimeZoneId", "unknown")
        notify = schedule.get("notifyOption", "unknown")

        if enabled:
            print(f"  \u2713 Enabled")
            print(f"  Days:     {', '.join(days) if days else '(not set)'}")
            print(f"  Times:    {', '.join(times) if times else '(not set)'}")
            print(f"  Timezone: {tz}")
            print(f"  Notify:   {notify}")
        else:
            print(f"  \u2717 Disabled (no schedule configured)")
    else:
        print(f"  \u26a0\ufe0f Could not retrieve schedule (API error or model not refreshable)")

## 4. Set refresh schedule

Configure the schedule below. The API requires:
- **enabled**: `True` to turn on scheduled refresh
- **days**: list of day names, e.g. `["Monday", "Wednesday", "Friday"]` or all 7 for daily
- **times**: list of HH:MM strings in 24h format, e.g. `["06:00", "18:00"]`
- **localTimeZoneId**: Windows timezone ID, e.g. `"AUS Eastern Standard Time"`, `"UTC"`, `"E. Australia Standard Time"`
- **notifyOption**: `"MailOnFailure"` or `"NoNotification"`

**Important:** The `PATCH .../refreshSchedule` API only works for the **dataset owner**.
If models are owned by a Service Principal, set `AUTO_TAKEOVER = True` to automatically
take over ownership before setting the schedule. Failure emails are sent to the owner.

**Limits:** Up to 8 refreshes/day on shared capacity, 48/day on Premium/F capacity.

In [ ]:
# === SCHEDULE CONFIGURATION ===

# Which models to set the schedule on — use names from the list above
# Set to None to apply to ALL refreshable models
SCHEDULE_MODEL_NAMES = None  # e.g., ["Finance Summary", "SalesAnalyticsModel"]

# Email address(es) to notify on scheduled refresh failure
# This sets the Fabric "Send refresh failure notification to" contacts
NOTIFY_EMAILS = ["data-team@company.com"]  # list of email addresses

# Auto-takeover: automatically take over SP-owned models before setting schedule
# The refreshSchedule API requires you to be the dataset owner
AUTO_TAKEOVER = True

# Schedule definition
SCHEDULE = {
    "enabled": True,
    "days": ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday"],
    "times": ["06:00", "12:00", "18:00"],
    "localTimeZoneId": "AUS Eastern Standard Time",
    "notifyOption": "MailOnFailure"
}

print("Schedule to apply:")
print(json.dumps(SCHEDULE, indent=2))
print(f"\nNotify on failure: {NOTIFY_EMAILS}")
print(f"Auto-takeover SP-owned models: {AUTO_TAKEOVER}")
print(f"Targeting: {'ALL refreshable models' if SCHEDULE_MODEL_NAMES is None else SCHEDULE_MODEL_NAMES}")

In [ ]:
# Apply the schedule — auto-takeover SP-owned models if enabled
if SCHEDULE_MODEL_NAMES is not None:
    targets = [m for m in refreshable_models if m["name"] in SCHEDULE_MODEL_NAMES]
    not_found = set(SCHEDULE_MODEL_NAMES) - {m["name"] for m in targets}
    if not_found:
        print(f"\u26a0\ufe0f Models not found or not refreshable: {sorted(not_found)}")
else:
    targets = refreshable_models

print(f"Applying schedule to {len(targets)} model(s):\n")

success = 0
failed = 0
taken_over = []

for m in targets:
    name = m["name"]
    mid = m["id"]
    owner = m.get("configuredBy", "unknown")
    is_sp_owned = "@" not in owner
    print(f"  '{name}' (owner: {owner}):")

    # Take over if SP-owned (required — refreshSchedule API only works for the owner)
    if is_sp_owned and AUTO_TAKEOVER:
        print(f"    Taking over from SP...")
        takeover_resp = api_post(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/Default.TakeOver")
        if takeover_resp.status_code == 200:
            print(f"    \u2713 Takeover successful — you are now the owner")
            taken_over.append(name)
        else:
            print(f"    \u2717 Takeover failed ({takeover_resp.status_code}): {takeover_resp.text[:300]}")
            print(f"    Skipping schedule (must be owner to set schedule)")
            failed += 1
            continue
    elif is_sp_owned:
        print(f"    \u26a0\ufe0f SP-owned but AUTO_TAKEOVER is False — schedule will likely fail")

    # Set the refresh schedule
    payload = {"value": SCHEDULE}
    resp = api_patch(
        f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/refreshSchedule",
        body=payload
    )

    if resp.status_code == 200:
        print(f"    \u2713 Schedule set (notifyOption: MailOnFailure)")
        success += 1
    else:
        print(f"    \u2717 Schedule failed ({resp.status_code}): {resp.text[:300]}")
        failed += 1

print(f"\nDone — {success} schedule(s) updated, {failed} failed")
if taken_over:
    print(f"\u2713 Took over {len(taken_over)} model(s): {', '.join(taken_over)}")
    print(f"\u26a0\ufe0f Data source credentials may need re-binding for taken-over models")

if NOTIFY_EMAILS:
    print(f"\n\u2139\ufe0f MailOnFailure sends to the dataset owner's email address.")
    print(f"  To add extra contacts: Fabric portal \u2192 Semantic model \u2192 Settings \u2192 Refresh")
    print(f"  \u2192 'Send failure notification to' \u2192 add: {', '.join(NOTIFY_EMAILS)}")

## 5. View refresh history

Check recent refresh runs to verify schedules are working or diagnose failures.

In [ ]:
# Show last 5 refreshes for each refreshable model
HISTORY_COUNT = 5

for m in refreshable_models:
    name = m["name"]
    mid = m["id"]

    history = api_get(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/refreshes?$top={HISTORY_COUNT}")
    entries = history.get("value", []) if history else []

    print(f"\n{name} — last {len(entries)} refresh(es):")
    print(f"  {'Status':<12} {'Type':<12} {'Start Time':<22} {'End Time':<22} {'Duration'}")
    print(f"  {'-'*85}")

    for entry in entries:
        status = entry.get("status", "Unknown")
        rtype = entry.get("refreshType", "?")
        start = entry.get("startTime", "")
        end = entry.get("endTime", "")

        # Calculate duration
        duration = ""
        if start and end:
            try:
                fmt = "%Y-%m-%dT%H:%M:%S.%fZ" if "." in start else "%Y-%m-%dT%H:%M:%SZ"
                s = datetime.strptime(start[:26] + "Z", fmt)
                e = datetime.strptime(end[:26] + "Z", fmt)
                secs = int((e - s).total_seconds())
                duration = f"{secs // 60}m {secs % 60}s"
            except ValueError:
                duration = "?"

        icon = "\u2713" if status == "Completed" else "\u2717" if status == "Failed" else "\u27a1"
        start_short = start[:19].replace("T", " ") if start else ""
        end_short = end[:19].replace("T", " ") if end else ""
        print(f"  {icon} {status:<10} {rtype:<12} {start_short:<22} {end_short:<22} {duration}")

        if status == "Failed":
            error = entry.get("serviceExceptionJson", "")
            if error:
                print(f"    Error: {error[:200]}")

## 6. Takeover ownership (optional)

If a model is owned by a Service Principal, you can take it over to a user account
so the refresh schedule is visible in the Fabric UI.

**Note:** The caller (whoever is running this notebook) becomes the new owner.
This means the model's data source credentials will need to be re-bound
to the new owner's identity. Only do this if intentional.

In [ ]:
# Set to the model names you want to take over (empty list = skip)
TAKEOVER_MODEL_NAMES = []  # e.g., ["Finance Summary"]

if not TAKEOVER_MODEL_NAMES:
    print("TAKEOVER_MODEL_NAMES is empty — skipping takeover.")
    print("To take over a model, add its name to the list and re-run this cell.")
else:
    takeover_targets = [m for m in all_models if m["name"] in TAKEOVER_MODEL_NAMES]
    not_found = set(TAKEOVER_MODEL_NAMES) - {m["name"] for m in takeover_targets}
    if not_found:
        print(f"\u26a0\ufe0f Models not found: {sorted(not_found)}")

    for m in takeover_targets:
        name = m["name"]
        mid = m["id"]
        current_owner = m.get("configuredBy", "unknown")
        print(f"\nTaking over '{name}' (current owner: {current_owner})...")

        resp = api_post(f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/Default.TakeOver")
        if resp.status_code == 200:
            print(f"  \u2713 Takeover successful — you are now the owner")
            print(f"  \u26a0\ufe0f Remember to re-bind data source credentials for this model")
        else:
            print(f"  \u2717 Takeover failed ({resp.status_code}): {resp.text[:300]}")

## 7. Disable refresh schedule (optional)

Run this cell to turn off scheduled refresh for specific models.

In [ ]:
# Set to the model names to disable (empty list = skip)
DISABLE_MODEL_NAMES = []  # e.g., ["SalesAnalyticsModel"]

if not DISABLE_MODEL_NAMES:
    print("DISABLE_MODEL_NAMES is empty — skipping.")
else:
    disable_targets = [m for m in refreshable_models if m["name"] in DISABLE_MODEL_NAMES]

    for m in disable_targets:
        name = m["name"]
        mid = m["id"]
        print(f"  Disabling schedule for '{name}'...")

        payload = {"value": {"enabled": False}}
        resp = api_patch(
            f"{PBI_API}/groups/{WORKSPACE_ID}/datasets/{mid}/refreshSchedule",
            body=payload
        )
        if resp.status_code == 200:
            print(f"    \u2713 Schedule disabled")
        else:
            print(f"    \u2717 Failed ({resp.status_code}): {resp.text[:300]}")